<a href="https://colab.research.google.com/github/LCaravaggio/FelicidadDesigualdad/blob/main/Resnet18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cargar Base

In [1]:
from google.colab import userdata
import json

!mkdir ~/.kaggle
!touch ~/.kaggle/kaggle.json

api_token = {
    'username': userdata.get('KAGGLE_USER'),
    'key': userdata.get('KAGGLE_KEY')}
with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(api_token, file)

!chmod 600 ~/.kaggle/kaggle.json


import kagglehub
path = kagglehub.dataset_download("leonardocaravaggio/ge-images")

100%|██████████| 13.3G/13.3G [03:06<00:00, 76.5MB/s]

Extracting files...


In [2]:
import pandas as pd
ciudades=pd.read_csv("base.csv")

In [3]:
len(ciudades)

1095

# Resnet18

In [602]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import torch.nn.functional as F


# Cargar MobileNetV3-Large preentrenada
# Cargar MobileNetV3-Large preentrenada
full_model = models.mobilenet_v3_large(pretrained=True)
mobilenet_v3 = full_model.features  # solo las capas convolucionales
mobilenet_v3.eval()


# Transformaciones para la imagen
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Redimensionar la imagen al tamaño esperado por MobileNetV3
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4, 0.4, 0.4],
                         std=[0.229, 0.224, 0.225]),  # Normalización estándar
])

def extract_features(image_path):
    """Extrae características de una imagen usando MobileNetV3-Large"""
    image = Image.open(image_path).convert("RGB")  # Cargar imagen
    image = transform(image).unsqueeze(0)  # Aplicar transformaciones y agregar batch dimension

    with torch.no_grad():
        features = mobilenet_v3(image)  # Extraer características
        pooled = F.adaptive_avg_pool2d(features, (4, 4))
        features = pooled.view(-1).numpy()

    return features

In [543]:
import numpy as np


def compute_inequality(image_1km_path, image_10km_path):
    # Extraer características de ambas imágenes
    features_1km = np.var(extract_features(image_1km_path)[:])
    features_10km = np.var(extract_features(image_10km_path)[:])

    # Calcular la diferencia entre ambas medidas de desigualdad
    difference = abs(features_1km - features_10km)

    return {
        "Desigualdad cercana (1km)": features_1km,
        "Desigualdad amplia (10km)": features_10km,
        "Diferencia entre ambas": difference
    }

In [608]:
ciudades['Desigualdad_10km']=np.nan
ciudades['Desigualdad_1km']=np.nan
ciudades['Diferencia']=np.nan

In [609]:
import os

for i in range(1095):
  if pd.isna(ciudades.loc[i, "Diferencia"]):  # Solo procesar si "Diferencia" está vacío
    try:
        nombre_archivo = ciudades.City[i].replace("/",".").replace(":","_").replace("'","!")
        ruta_completa = os.path.join(path, "imagenes", nombre_archivo)

        if not os.path.exists(ruta_completa + " - 1K.png"):
          print(ruta_completa + "no existe")


        resultados = compute_inequality(ruta_completa + " - 1K.png",
                                        ruta_completa + " - 10K.png")
        ciudades.loc[i, "Desigualdad_1km"] = resultados["Desigualdad cercana (1km)"]
        ciudades.loc[i, "Desigualdad_10km"] = resultados["Desigualdad amplia (10km)"]
        ciudades.loc[i, "Diferencia"] = resultados["Diferencia entre ambas"]
    except Exception as e:
        print(f"⚠️ Error en compute_inequality: {e}")

In [603]:
compute_inequality('/content/Oceano - 1K.png', '/content/Oceano - 10K.png')

{'Desigualdad cercana (1km)': np.float32(0.066856764),
 'Desigualdad amplia (10km)': np.float32(0.0663),
 'Diferencia entre ambas': np.float32(0.000556767)}

In [604]:
compute_inequality('/content/Amazonas - 1K.png', '/content/Amazonas - 10K.png')

{'Desigualdad cercana (1km)': np.float32(0.10225137),
 'Desigualdad amplia (10km)': np.float32(0.2519152),
 'Diferencia entre ambas': np.float32(0.1496638)}

In [605]:
compute_inequality('/content/Favela Rocinha - 1K.png', '/content/Favela Rocinha - 10K.png')

{'Desigualdad cercana (1km)': np.float32(0.38724074),
 'Desigualdad amplia (10km)': np.float32(0.52004886),
 'Diferencia entre ambas': np.float32(0.13280812)}

In [606]:
compute_inequality('/content/Retiro - 1K.png', '/content/Retiro - 10K.png')

{'Desigualdad cercana (1km)': np.float32(0.46376994),
 'Desigualdad amplia (10km)': np.float32(0.49039525),
 'Diferencia entre ambas': np.float32(0.026625305)}

In [607]:
vitacura='/root/.cache/kagglehub/datasets/leonardocaravaggio/ge-images/versions/2/imagenes/CL_ Metropolitana-Vitacura'
compute_inequality(vitacura+' - 1K.png', vitacura+' - 10K.png')

{'Desigualdad cercana (1km)': np.float32(0.35492888),
 'Desigualdad amplia (10km)': np.float32(0.3731469),
 'Diferencia entre ambas': np.float32(0.01821801)}

# Bajar la base con el indicador de desigualdad

In [572]:
from google.colab import files
name="base_mobilv3_norm.csv"
ciudades.to_csv(name)
files.download(name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [611]:
import statsmodels.api as sm
import numpy as np

# Definir variables
X = ciudades["Desigualdad_10km"].replace([np.inf, -np.inf], np.nan)
y = ciudades["P1ST"].replace([np.inf, -np.inf], np.nan)

# Filtrar filas con NaN en X o y
mask = X.notna() & y.notna()
X, y = X[mask], y[mask]

# Agregar constante para la ordenada al origen
X = sm.add_constant(X)

# Ajustar modelo
modelo = sm.OLS(y, X).fit()

# Resumen de la regresión
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                   P1ST   R-squared:                       0.024
Model:                            OLS   Adj. R-squared:                  0.023
Method:                 Least Squares   F-statistic:                     27.01
Date:                Fri, 04 Apr 2025   Prob (F-statistic):           2.41e-07
Time:                        21:14:38   Log-Likelihood:                -474.64
No. Observations:                1095   AIC:                             953.3
Df Residuals:                    1093   BIC:                             963.3
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const                3.3438      0.046  